In [ ]:
# Fix protobuf version mismatch for kaggle_benchmarks
!pip install protobuf==5.29.6 --quiet


# Trinity Executive Function Battery — Stroop-like Inhibition

**Task 17 of 25** | Track 4: Executive | Brain Zone: PALLIDUS

This notebook measures inhibition ability.

## Trinity Neuroanatomical Foundation

PALLIDUS implements GABA gate via OFC.checkDestructiveness().

### Ternary Scoring {-1, 0, +1}
- **+1**: Correct inhibition
- **0**: Partial with uncertainty
- **-1**: Failed inhibition

### φ-Scaling
Inhibition cost: 1, 2, 3, 4, 5

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

df = pd.read_csv('../../data/tefb_executive.csv')
stroop_df = df[df['task'] == 'Stroop-like Inhibition'].copy()


In [ ]:
@dataclass
class InhibitionResponse:
    did_inhibit: bool
    response: str
    confidence: float

    def ternary_score(self, expected_behavior: str) -> int:
        if self.did_inhibit and expected_behavior == "Inhibit":
            return 1
        elif 0.3 < self.confidence < 0.7:
            return 0
        else:
            return -1


In [ ]:
@kbench.task(name="trinity_pallidus_inhibition")
def stroop_inhibition(llm, automatic_response, instruction, target, inhibition_cost) -> float:
    prompt = f""Automatic response: {automatic_response}\n\nInstruction: {instruction}\n\nTarget: {target}\n\nDo you follow instruction or respond automatically?\n\nConfidence (0-1)"

    response = llm.prompt(prompt, schema=InhibitionResponse)
    ternary = response.ternary_score("Inhibit")
    accuracy = 1.0 if (response.did_inhibit and target == "Inhibit") else -1.0
    confidence_score = (response.confidence - 0.5) * 2
    final_score = 0.7 * accuracy + 0.3 * confidence_score

    return max(-1.0, min(1.0, final_score))

print("Task registered")

In [ ]:
results = stroop_inhibition.evaluate(llm=[kbench.llm], evaluation_data=stroop_df.head(10))
print(results.head())

In [ ]:
# Full eval
# results = stroop_inhibition.evaluate(llm=[kbench.llm], evaluation_data=stroop_df)
# kbench.submit(task=stroop_inhibition, results=results, message="Trinity Pallidus Stroop v1.0")
# print("Submitted!")